# AutoML · M07: Comparativa Final

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | AutoML — Pre-Modelado |
| **Módulo** | M07 — Comparativa Final |

---

## 🎯 Qué hace

Consolida los resultados de todos los frameworks AutoML (M01-M06) en un
único ranking global. Identifica el mejor modelo de cada framework,
establece el baseline definitivo que Fase 5 debe superar y documenta
las decisiones metodológicas del screening.

## 📋 Requisitos

- Todos los `results_*.parquet` en `data/automl/` (M01-M06 ejecutados)
- Entorno: `tfm_abandono`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/automl/automl_comparativa_final.xlsx` | Excel con ranking completo |
| `data/automl/automl_top_modelos.parquet` | Mejor modelo por framework |
| `docs/html/fase_automl/m07_comparativa.html` | Informe HTML interactivo |

## 🔄 Flujo

```
results_baselines.parquet    (M01)
results_lazypredict.parquet  (M02)
results_pycaret.parquet      (M03)
results_h2o.parquet          (M04)
results_autogluon.parquet    (M05)
results_tabpfn.parquet       (M06)
    ↓ Consolidar + ranking global
    → Excel + Parquet + HTML
```

## ➡️ Siguiente

`fautoml_m00_indice.ipynb` — índice de la fase AutoML


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Detectar ROOT robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_AUTOML, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html import generar_kpis_html, generar_seccion_html
from src.html.render import render_pagina

RUTA_FASE_AUTOML_HTML = RUTA_HTML / 'fase_automl'
crear_directorios([RUTA_FASE_AUTOML_HTML])

fmt           = formato_numero_es
NOTEBOOK_NAME = 'fautoml_m07_comparativa.ipynb'
CARPETA_NB    = 'fase_automl'

# Colores por framework
COLORES_FW = {
    'baselines'  : '#3182ce',
    'lazypredict': '#d69e2e',
    'pycaret'    : '#38a169',
    'h2o'        : '#805ad5',
    'autogluon'  : '#ed8936',
    'tabpfn'     : '#e53e3e',
}

info_entorno()
print('\n✅ Configuración completada')


✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR Y CONSOLIDAR TODOS LOS RESULTADOS
# ============================================================================
# Carga todos los results_*.parquet de M01-M06.
# Los parquets nuevos no tienen columna 'caso' — se añade 'D_strict' por defecto.
# TabPFN se trata por separado por su naturaleza distinta (dataset completo, umbral 0.5).
# ============================================================================

print('\n' + '='*70)
print('CARGANDO RESULTADOS DE TODOS LOS FRAMEWORKS')
print('='*70)

# Mapeo de ficheros a etiqueta limpia
FICHEROS = [
    ('results_baselines.parquet',   'M01 Baselines',   True),
    ('results_lazypredict.parquet', 'M02 LazyPredict', False),
    ('results_pycaret.parquet',     'M03 PyCaret',     True),
    ('results_h2o.parquet',         'M04 H2O',         True),
    ('results_autogluon.parquet',   'M05 AutoGluon',   True),
    ('results_tabpfn.parquet',      'M06 TabPFN v2',   True),
]

dfs = []
for fname, etiqueta, tiene_cv in FICHEROS:
    ruta = RUTA_AUTOML / fname
    if not ruta.exists():
        print(f'  ⚠️  {fname} no encontrado — omitido')
        continue
    df = pd.read_parquet(ruta)
    # Añadir columna modulo para identificar el origen
    df['modulo']   = etiqueta
    df['tiene_cv'] = tiene_cv
    # Normalizar framework a minúsculas
    df['framework'] = df['framework'].str.lower().str.strip()
    dfs.append(df)
    print(f'  ✅ {fname}: {len(df)} modelos ({etiqueta})')

if not dfs:
    raise FileNotFoundError('No hay results_*.parquet. Ejecutar M01-M06 primero.')

df_all = pd.concat(dfs, ignore_index=True)

# Excluir Dummy para el ranking real
df_real = df_all[~df_all['model_name'].str.startswith('Dummy')].copy()

print(f'\n📊 Total modelos: {len(df_all)} ({len(df_real)} sin Dummy)')
print(f'   Frameworks: {sorted(df_all["framework"].unique())}')
print(f'   Módulos: {sorted(df_all["modulo"].unique())}')



CARGANDO RESULTADOS DE TODOS LOS FRAMEWORKS


  ✅ results_baselines.parquet: 6 modelos (M01 Baselines)
  ✅ results_lazypredict.parquet: 27 modelos (M02 LazyPredict)


  ✅ results_pycaret.parquet: 15 modelos (M03 PyCaret)
  ✅ results_h2o.parquet: 21 modelos (M04 H2O)
  ✅ results_autogluon.parquet: 10 modelos (M05 AutoGluon)
  ✅ results_tabpfn.parquet: 1 modelos (M06 TabPFN v2)

📊 Total modelos: 80 (76 sin Dummy)
   Frameworks: ['autogluon', 'baselines', 'h2o', 'lazypredict', 'pycaret', 'tabpfn']
   Módulos: ['M01 Baselines', 'M02 LazyPredict', 'M03 PyCaret', 'M04 H2O', 'M05 AutoGluon', 'M06 TabPFN v2']


In [3]:
# ============================================================================
# CELDA 3: RANKINGS Y MEJOR POR FRAMEWORK
# ============================================================================

# Ranking global por F1
df_ranking = df_real.sort_values('f1', ascending=False).reset_index(drop=True)

print('\n--- RANKING GLOBAL TOP 15 (por F1) ---')
cols_show = ['modulo', 'model_name', 'f1', 'auc_roc', 'mcc', 'precision', 'recall']
print(df_ranking[cols_show].head(15).to_string(index=False))

# Mejor modelo por framework
df_mejor = (
    df_real
    .sort_values('f1', ascending=False)
    .groupby('modulo', sort=False)
    .first()
    .reset_index()
    .sort_values('f1', ascending=False)
)

print('\n--- MEJOR MODELO POR FRAMEWORK ---')
print(df_mejor[['modulo', 'model_name', 'f1', 'auc_roc', 'mcc', 'tiene_cv']].to_string(index=False))

# Mejor global
mejor_global = df_ranking.iloc[0]
nombre_mejor = mejor_global['model_name']
print(f'\n🏆 MEJOR GLOBAL: {mejor_global["modulo"]} — {nombre_mejor}')
print(f'   F1={mejor_global["f1"]:.4f} | AUC={mejor_global["auc_roc"]:.4f} | MCC={mejor_global["mcc"]:.4f}')



--- RANKING GLOBAL TOP 15 (por F1) ---
         modulo                      model_name       f1  auc_roc      mcc  precision   recall
  M06 TabPFN v2                       TabPFN v2 0.849564 0.964380 0.793653   0.894638 0.808814
  M05 AutoGluon             WeightedEnsemble_L2 0.832976 0.957994 0.771430   0.881529 0.789492
  M05 AutoGluon                 CatBoost_BAG_L1 0.832829 0.956395 0.770427   0.877015 0.792881
  M05 AutoGluon                 LightGBM_BAG_L1 0.832267 0.957108 0.769307   0.874533 0.793898
  M05 AutoGluon                  XGBoost_BAG_L1 0.832088 0.956411 0.769243   0.875374 0.792881
  M05 AutoGluon               LightGBMXT_BAG_L1 0.831673 0.954056 0.768729   0.875281 0.792203
  M01 Baselines                         XGBoost 0.829648 0.953174 0.764915   0.867555 0.794915
M02 LazyPredict                   XGBClassifier 0.829648 0.872377 0.764915   0.867555 0.794915
        M04 H2O                        GBM_grid 0.829554 0.954411 0.767226   0.872979 0.790244
        M0

In [4]:
# ============================================================================
# CELDA 4: GRÁFICOS
# ============================================================================
# Gráfico 1: Top 15 modelos globales — barras horizontales con color por framework.
# Gráfico 2: Mejor F1 por framework — comparativa directa entre módulos.
# Gráfico 3: F1 vs AUC scatter — todos los modelos.
# ============================================================================

graficos_b64 = {}

# --- Gráfico 1: Top 15 global ---
df_top15 = df_ranking.head(15).sort_values('f1', ascending=True).copy()
colores_barras = [COLORES_FW.get(fw, '#718096') for fw in df_top15['framework']]

fig, ax = plt.subplots(figsize=(13, 8))
y_pos = np.arange(len(df_top15))
bars  = ax.barh(y_pos, df_top15['f1'], color=colores_barras, alpha=0.85, height=0.6)
ax.scatter(df_top15['auc_roc'], y_pos, color='black', s=50, zorder=5, label='AUC-ROC')
ax.set_yticks(y_pos)
ax.set_yticklabels(
    [f"{r['modulo']}: {r['model_name'][:30]}" for _, r in df_top15.iterrows()],
    fontsize=8
)
ax.set_xlabel('Score')
ax.set_title('Top 15 Modelos — Ranking Global AutoML (todos los frameworks)',
             fontweight='bold', fontsize=13)
ax.set_xlim(0.7, 1.0)
ax.legend(fontsize=9)
for bar, val in zip(bars, df_top15['f1']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)

# Leyenda de colores por framework
from matplotlib.patches import Patch
leyenda_fw = [Patch(color=v, label=k) for k, v in COLORES_FW.items() if k in df_top15['framework'].values]
ax.legend(handles=leyenda_fw + [plt.scatter([], [], color='black', s=50, label='AUC-ROC')],
          fontsize=8, loc='lower right')
plt.tight_layout()
graficos_b64['top15'] = figura_a_base64(fig)
plt.close()

# --- Gráfico 2: Mejor F1 por framework ---
fig, ax = plt.subplots(figsize=(11, 5))
modulos  = df_mejor['modulo'].tolist()
f1s      = df_mejor['f1'].tolist()
aucs     = df_mejor['auc_roc'].tolist()
fw_names = df_mejor['framework'].tolist()
colores_fw_bar = [COLORES_FW.get(fw, '#718096') for fw in fw_names]

x     = np.arange(len(modulos))
width = 0.35
bars_f1  = ax.bar(x - width/2, f1s,  width, color=colores_fw_bar, alpha=0.85, label='F1')
bars_auc = ax.bar(x + width/2, aucs, width, color=colores_fw_bar, alpha=0.45, label='AUC-ROC',
                  edgecolor=colores_fw_bar, linewidth=1.5)

ax.set_xticks(x)
ax.set_xticklabels(modulos, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('Mejor F1 y AUC por framework AutoML', fontweight='bold', fontsize=13)
ax.set_ylim(0.7, 1.0)
ax.axhline(y=0.83, color='gray', ls='--', alpha=0.4, label='F1=0.83 referencia')
ax.legend(fontsize=9)

for bar, val in zip(bars_f1, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
graficos_b64['mejor_fw'] = figura_a_base64(fig)
plt.close()

# --- Gráfico 3: F1 vs AUC scatter ---
fig, ax = plt.subplots(figsize=(10, 7))
for fw, color in COLORES_FW.items():
    mask = df_real['framework'] == fw
    if mask.sum() == 0:
        continue
    df_fw = df_real[mask]
    modulo_label = df_fw['modulo'].iloc[0]
    ax.scatter(df_fw['auc_roc'], df_fw['f1'],
               color=color, alpha=0.7, s=60, label=modulo_label)

# Marcar el mejor global
ax.scatter(mejor_global['auc_roc'], mejor_global['f1'],
           color='gold', s=200, zorder=10, marker='*', edgecolors='black', linewidth=1,
           label=f'Mejor: {mejor_global["modulo"]}')

ax.set_xlabel('AUC-ROC')
ax.set_ylabel('F1')
ax.set_title('F1 vs AUC-ROC — todos los modelos AutoML', fontweight='bold', fontsize=13)
ax.legend(fontsize=8, loc='lower right')
ax.set_xlim(0.5, 1.0)
ax.set_ylim(0.0, 1.0)
ax.axhline(y=0.83, color='gray', ls='--', alpha=0.3)
ax.axvline(x=0.93, color='gray', ls='--', alpha=0.3)
plt.tight_layout()
graficos_b64['scatter'] = figura_a_base64(fig)
plt.close()

print(f'✅ {len(graficos_b64)} gráficos generados')


✅ 3 gráficos generados


In [5]:
# ============================================================================
# CELDA 5: GUARDAR RESULTADOS
# ============================================================================

# Excel con hojas: ranking global + mejor por framework
ruta_xlsx = RUTA_AUTOML / 'automl_comparativa_final.xlsx'
with pd.ExcelWriter(ruta_xlsx, engine='openpyxl') as writer:
    df_ranking.to_excel(writer, sheet_name='ranking_global', index=False)
    df_mejor.to_excel(writer, sheet_name='mejor_por_framework', index=False)
    df_all.to_excel(writer, sheet_name='todos_resultados', index=False)
print(f'📊 Excel: {ruta_xlsx.name}')

# Parquet top modelos
ruta_top = RUTA_AUTOML / 'automl_top_modelos.parquet'
df_mejor.to_parquet(ruta_top, index=False)
print(f'📊 Top modelos: {ruta_top.name}')


📊 Excel: automl_comparativa_final.xlsx
📊 Top modelos: automl_top_modelos.parquet


In [6]:
# ============================================================================
# CELDA 6: GENERAR HTML
# ============================================================================

# --- KPIs ---
n_frameworks = df_real['framework'].nunique()
n_modelos    = len(df_real)
nombre_mejor = mejor_global['model_name']
modulo_mejor = mejor_global['modulo']

KPIS = [
    {'valor': str(n_modelos),                  'titulo': 'Modelos evaluados'},
    {'valor': str(n_frameworks),               'titulo': 'Frameworks'},
    {'valor': f'{mejor_global["f1"]:.4f}',     'titulo': f'Mejor F1'},
    {'valor': f'{mejor_global["auc_roc"]:.4f}','titulo': f'Mejor AUC'},
]

# --- Tabla ranking global ---
tabla = '<table style="width:100%;border-collapse:collapse;font-size:11px;">\n'
tabla += '<tr style="background:#2d3748;color:white;">'
for col in ['#', 'Módulo', 'Modelo', 'F1', 'AUC', 'MCC', 'Precision', 'Recall', 'CV']:
    tabla += f'<th style="padding:8px;text-align:center;">{col}</th>'
tabla += '</tr>\n'

for rank, (_, row) in enumerate(df_ranking.head(30).iterrows(), 1):
    bg = '#f7fafc' if rank % 2 == 0 else 'white'
    medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
    rank_str    = medallas.get(rank, str(rank))
    nombre_mod  = row['model_name']
    modulo_str  = row['modulo']
    color_fw    = COLORES_FW.get(row['framework'], '#718096')
    cv_str      = 'CV ✅' if row['tiene_cv'] else 'Sin CV'
    tabla += f'<tr style="background:{bg};">'
    tabla += f'<td style="padding:6px;text-align:center;">{rank_str}</td>'
    tabla += f'<td style="padding:6px;color:{color_fw};font-weight:bold;">{modulo_str}</td>'
    tabla += f'<td style="padding:6px;">{nombre_mod}</td>'
    for campo in ['f1', 'auc_roc', 'mcc', 'precision', 'recall']:
        v = row.get(campo, 0)
        if pd.isna(v): v = 0
        color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
        tabla += f'<td style="text-align:center;color:{color};">{v:.4f}</td>'
    tabla += f'<td style="text-align:center;font-size:10px;">{cv_str}</td>'
    tabla += '</tr>\n'
tabla += '</table>'
s_tabla = generar_seccion_html('Ranking global — Top 30 modelos (por F1)', tabla, '🏆')

# --- Tabla mejor por framework ---
tabla_fw = '<table style="width:100%;border-collapse:collapse;font-size:12px;">\n'
tabla_fw += '<tr style="background:#2d3748;color:white;">'
for col in ['Módulo', 'Mejor Modelo', 'F1', 'AUC', 'MCC', 'CV', 'Tiempo']:
    tabla_fw += f'<th style="padding:10px;text-align:center;">{col}</th>'
tabla_fw += '</tr>\n'

for rank, (_, row) in enumerate(df_mejor.iterrows(), 1):
    bg = '#f7fafc' if rank % 2 == 0 else 'white'
    medallas = {1: '🥇', 2: '🥈', 3: '🥉'}
    rank_str   = medallas.get(rank, '')
    color_fw   = COLORES_FW.get(row['framework'], '#718096')
    cv_str     = 'CV=5 ✅' if row['tiene_cv'] else 'Sin CV ⚠️'
    nombre_mod = row['model_name']
    modulo_str = row['modulo']
    tabla_fw += f'<tr style="background:{bg};">'
    tabla_fw += f'<td style="padding:8px;color:{color_fw};font-weight:bold;">{rank_str} {modulo_str}</td>'
    tabla_fw += f'<td style="padding:8px;">{nombre_mod}</td>'
    for campo in ['f1', 'auc_roc', 'mcc']:
        v = row.get(campo, 0)
        if pd.isna(v): v = 0
        color = '#38a169' if v > 0.7 else '#ed8936' if v > 0.5 else '#e53e3e'
        tabla_fw += f'<td style="text-align:center;font-weight:bold;color:{color};">{v:.4f}</td>'
    tabla_fw += f'<td style="text-align:center;">{cv_str}</td>'
    t = row.get('train_time_sec', 0)
    tabla_fw += f'<td style="text-align:center;">{t:.1f}s</td>'
    tabla_fw += '</tr>\n'
tabla_fw += '</table>'
s_tabla_fw = generar_seccion_html('Mejor modelo por framework', tabla_fw, '📊')

# --- Gráficos ---
def img(key):
    return f'<img src="data:image/png;base64,{graficos_b64[key]}" style="max-width:100%;border-radius:8px;">'

s_graf1 = generar_seccion_html('Top 15 modelos — ranking global', img('top15'), '📈')
s_graf2 = generar_seccion_html('Mejor F1 y AUC por framework', img('mejor_fw'), '📊')
s_graf3 = generar_seccion_html('F1 vs AUC-ROC — todos los modelos', img('scatter'), '🔵')

# --- Conclusión metodológica ---
mejor_f1_val  = mejor_global['f1']
mejor_auc_val = mejor_global['auc_roc']
mejor_mcc_val = mejor_global['mcc']
s_concl = generar_seccion_html('Conclusión y baseline para Fase 5', f'''
<div style="background:#f0fff4;padding:20px;border-radius:10px;
            border-left:4px solid #38a169;margin-bottom:15px;">
  <h4 style="color:#38a169;margin-top:0;">🏆 Mejor modelo del screening AutoML</h4>
  <p><strong>{modulo_mejor} — {nombre_mejor}</strong></p>
  <p>F1 = {mejor_f1_val:.4f} | AUC = {mejor_auc_val:.4f} | MCC = {mejor_mcc_val:.4f}</p>
  <p>Este resultado es el <strong>baseline que Fase 5 debe superar</strong>.
  Los modelos de Fase 5 con tuning manual y ensambles avanzados deben alcanzar
  al menos F1 &gt; {mejor_f1_val:.4f} para justificar el esfuerzo adicional.</p>
</div>
<div style="background:#ebf8ff;padding:15px;border-radius:10px;
            border-left:4px solid #3182ce;margin-bottom:15px;">
  <strong>Nota sobre LazyPredict (M02):</strong> Sus métricas son orientativas
  — no realiza validación cruzada. Los resultados de M01, M03, M04 y M05
  son estadísticamente comparables (split 70/30 + CV o bagging).
</div>
<div style="background:#fffaf0;padding:15px;border-radius:10px;
            border-left:4px solid #ed8936;">
  <strong>Nota sobre TabPFN v2 (M06):</strong> Evaluado sobre el dataset
  completo con split 70/30 y umbral 0.5, igual que el resto. Sin
  hiperparámetros — baseline "zero-shot" de referencia.
</div>''', '📌')

# --- Render HTML ---
ruta_html = RUTA_FASE_AUTOML_HTML / 'm07_comparativa.html'
render_pagina(
    NOTEBOOK_NAME,
    generar_kpis_html(KPIS) + s_tabla_fw + s_graf1 + s_graf2 + s_graf3 + s_tabla + s_concl,
    ruta_html,
    carpeta_notebook=CARPETA_NB
)
print(f'\n✅ HTML: {ruta_html}')



✅ HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m07_comparativa.html


In [7]:
# ============================================================================
# CELDA 7: RESUMEN FINAL
# ============================================================================

print('\n' + '='*60)
print('✅ AUTOML-M07 COMPLETADO')
print('='*60)
print(f'Total modelos evaluados : {n_modelos}')
print(f'Frameworks              : {n_frameworks}')
print(f'\n🏆 Mejor modelo global  : {modulo_mejor} — {nombre_mejor}')
print(f'   F1  = {mejor_global["f1"]:.4f}')
print(f'   AUC = {mejor_global["auc_roc"]:.4f}')
print(f'   MCC = {mejor_global["mcc"]:.4f}')
print(f'\n📊 Archivos generados:')
print(f'   Excel    : {RUTA_AUTOML / "automl_comparativa_final.xlsx"}')
print(f'   Parquet  : {RUTA_AUTOML / "automl_top_modelos.parquet"}')
print(f'   HTML     : {RUTA_FASE_AUTOML_HTML / "m07_comparativa.html"}')
print(f'\n➡️  Siguiente : fautoml_m00_indice.ipynb')
print('='*60)



✅ AUTOML-M07 COMPLETADO
Total modelos evaluados : 76
Frameworks              : 6

🏆 Mejor modelo global  : M06 TabPFN v2 — TabPFN v2
   F1  = 0.8496
   AUC = 0.9644
   MCC = 0.7937

📊 Archivos generados:
   Excel    : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\automl_comparativa_final.xlsx
   Parquet  : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl\automl_top_modelos.parquet
   HTML     : C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase_automl\m07_comparativa.html

➡️  Siguiente : fautoml_m00_indice.ipynb
